# Prepare dataset from separated pickle files

You should have separate `.pkl` files named as `*AXXX.pkl`, where `XXX` is a label index. Files should include: 
```
{
    'keypoint': np.ndarray   # (num_person, num_frames, num_keypoints, 2)
    'keypoint_score': np.ndarray  # (num_person, num_frames, num_keypoints)
    'frame_dir': str          # file names without extension
    'img_shape': tuple        # (height, width)
    'original_shape': tuple   # (height, width)
    'total_frames': int       # number of frames
    'label': int              # class name (0-based)
}
```

In [ ]:
!pip install mmengine

In [2]:
import os
import pickle
import cv2
import numpy as np
from tqdm import tqdm
import json
import random
import mmengine
from collections import Counter, defaultdict

## Inspect pkl

In [3]:
def print_pkl(path, num_frames=3, num_persons=1, num_points=5):
    print(f"\nLoading: {path}")
    with open(path, "rb") as f:
        data = pickle.load(f)

    print("\n=== PKL Keys ===")
    for k in data.keys():
        print(" •", k)

    # Print shapes
    if "keypoint" in data:
        kp = data["keypoint"]
        print("\nkeypoint shape:", kp.shape,
              " -> (num_persons, num_frames, num_points, 2)")

    if "keypoint_score" in data:
        ks = data["keypoint_score"]
        print("keypoint_score shape:", ks.shape,
              " -> (num_persons, num_frames, num_points)")

    # Print simple fields
    for field in ["frame_dir", "img_shape", "original_shape", "total_frames", "label"]:
        if field in data:
            print(f"{field}: {data[field]}")

    print("\n=== Sample Values ===")
    try:
        kp = data["keypoint"]
        ks = data["keypoint_score"]

        P = min(num_persons, kp.shape[0])
        F = min(num_frames, kp.shape[1])
        J = min(num_points, kp.shape[2])

        for p in range(P):
            print(f"\n--- Person {p} ---")
            for f in range(F):
                print(f" Frame {f}:")
                for j in range(J):
                    print(f"   Joint {j}: xy={kp[p, f, j]},   score={ks[p, f, j]:.3f}")

    except Exception as e:
        print("Error while printing sample values:", e)


In [ ]:
#print_pkl("../../data/dataset/keypoints/2A001.pkl")


Loading: ../../data/dataset/keypoints/2A001.pkl

=== PKL Keys ===
 • keypoint
 • keypoint_score
 • frame_dir
 • img_shape
 • original_shape
 • total_frames
 • label

keypoint shape: (1, 47, 17, 2)  -> (num_persons, num_frames, num_points, 2)
keypoint_score shape: (1, 47, 17)  -> (num_persons, num_frames, num_points)
frame_dir: 2A001
img_shape: (1080, 1920)
original_shape: (1080, 1920)
total_frames: 47
label: 0

=== Sample Values ===

--- Person 0 ---
 Frame 0:
   Joint 0: xy=[330.9173      2.4288015],   score=0.940
   Joint 1: xy=[337.85672    -1.0409149],   score=0.849
   Joint 2: xy=[323.97784    -1.0409149],   score=0.862
   Joint 3: xy=[351.7356     9.368234],   score=0.943
   Joint 4: xy=[310.099      9.368234],   score=0.944
 Frame 1:
   Joint 0: xy=[329.75348    2.430246],   score=0.905
   Joint 1: xy=[340.16882    -1.0415428],   score=0.846
   Joint 2: xy=[322.80988    -1.0415428],   score=0.850
   Joint 3: xy=[354.056      9.373824],   score=0.907
   Joint 4: xy=[312.39453   

## Visualize pkl

In [ ]:
# Цвета для разных людей
COLORS = [(0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]

# Соединения ключевых точек для скелета COCO (17 keypoints)
SKELETON = [
    (0, 1), (0, 2),         # нос -> глаза
    (1, 3), (2, 4),         # глаза -> уши
    (0, 5), (0, 6),         # нос -> плечи
    (5, 7), (7, 9),         # левое плечо -> локоть -> кисть
    (6, 8), (8, 10),        # правое плечо -> локоть -> кисть
    (5, 11), (6, 12),       # плечи -> бедра
    (11, 13), (13, 15),     # левое бедро -> колено -> стопа
    (12, 14), (14, 16)      # правое бедро -> колено -> стопа
]


def draw_skeleton(frame, keypoints, color=(0, 255, 0), radius=4):
    """
    Рисует корректный COCO скелет на кадре
    keypoints: shape (num_keypoints, 2)
    """
    # рисуем точки
    for x, y in keypoints:
        if x > 0 and y > 0:
            cv2.circle(frame, (int(x), int(y)), radius, color, -1)

    # рисуем соединения
    for i, j in SKELETON:
        if keypoints[i, 0] > 0 and keypoints[i, 1] > 0 and \
           keypoints[j, 0] > 0 and keypoints[j, 1] > 0:
            cv2.line(frame, (int(keypoints[i, 0]), int(keypoints[i, 1])),
                     (int(keypoints[j, 0]), int(keypoints[j, 1])), color, 2)
    return frame


def visualize_pkl(video_path, pkl_path, output_path='output.avi'):
    # Загружаем аннотации
    anno = mmengine.load(pkl_path)
    keypoints = anno['keypoint']  # shape: (num_persons, num_frames, num_keypoints, 2)

    num_persons, num_frames, num_kpts, _ = keypoints.shape

    # Открываем видео
    cap = cv2.VideoCapture(video_path)
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame_idx >= num_frames:
            break

        for p_idx in range(num_persons):
            kpts = keypoints[p_idx, frame_idx]
            frame = draw_skeleton(frame, kpts, color=COLORS[p_idx % len(COLORS)])

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print(f'Визуализация сохранена: {output_path}')

In [ ]:
#visualize_pkl("../../data/dataset/raw/gesture1/1A001.avi", "../../data/dataset/keypoints/1A001.pkl", "../../data/dataset/results/1A001.avi")

Визуализация сохранена: ../../data/dataset/results/1A001.avi


## Clean pkl

Remove false detections

In [ ]:
def clean_pkl(path):
    with open(path, 'rb') as f:
        data = pickle.load(f)

    keypoint = data['keypoint']          # shape: (P, T, V, 2)
    keypoint_score = data['keypoint_score']  # shape: (P, T, V)

    num_persons, T, V, _ = keypoint.shape

    # Если персон 1 — ничего не делаем
    if num_persons <= 1:
        return False  # no changes

    # Вычисляем количество "валидных" кейпоинтов для каждого человека
    # valid = координаты не (0,0) И score > 0.05
    valid_counts = []
    for p in range(num_persons):
        coords = keypoint[p]             # (T, V, 2)
        scores = keypoint_score[p]       # (T, V)

        coord_valid = np.logical_or(coords[...,0] != 0, coords[...,1] != 0)
        score_valid = scores > 0.05

        person_valid = np.logical_and(coord_valid, score_valid)
        valid_counts.append(person_valid.sum())

    # Сортируем персон по количеству валидных точек
    best = np.argsort(valid_counts)[-1:]   # индекс лучшего

    # Оставляем
    data['keypoint'] = keypoint[best]
    data['keypoint_score'] = keypoint_score[best]

    # Сохраняем обратно
    with open(path, 'wb') as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

    return True  # changed


def process_folder_clean(folder):
    pkl_files = [f for f in os.listdir(folder) if f.endswith('.pkl')]

    for fname in tqdm(pkl_files, desc="Cleaning PKLs"):
        path = os.path.join(folder, fname)
        changed = clean_pkl(path)
        if changed:
            print(f"✔ Очистка выполнена для: {fname}")

In [ ]:
#process_folder_clean("../../data/dataset/keypoints")

Cleaning PKLs: 100%|██████████| 48/48 [00:00<00:00, 4850.66it/s]

✔ Очистка выполнена для: 5A002.pkl
✔ Очистка выполнена для: 12A005.pkl
✔ Очистка выполнена для: 16A005.pkl
✔ Очистка выполнена для: 5A003.pkl
✔ Очистка выполнена для: 3A001.pkl
✔ Очистка выполнена для: 7A004.pkl
✔ Очистка выполнена для: 14A005.pkl
✔ Очистка выполнена для: 9A005.pkl
✔ Очистка выполнена для: 6A004.pkl
✔ Очистка выполнена для: 2A001.pkl
✔ Очистка выполнена для: 13A005.pkl
✔ Очистка выполнена для: 10A005.pkl
✔ Очистка выполнена для: 4A004.pkl
✔ Очистка выполнена для: 8A005.pkl
✔ Очистка выполнена для: 7A005.pkl
✔ Очистка выполнена для: 4A003.pkl
✔ Очистка выполнена для: 1A005.pkl
✔ Очистка выполнена для: 3A005.pkl
✔ Очистка выполнена для: 8A003.pkl
✔ Очистка выполнена для: 2A005.pkl
✔ Очистка выполнена для: 3A004.pkl


## Statistics over folder

In [ ]:
def get_num_frames(pkl_path):
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)

    # keypoint shape: (P, T, V, C)
    keypoint = data['keypoint']
    _, T, _, _ = keypoint.shape
    return T


def process_folder_num_frames(folder):
    frame_counts = Counter()

    files = [f for f in os.listdir(folder) if f.endswith('.pkl')]

    for fname in tqdm(files, desc="Reading PKLs"):
        path = os.path.join(folder, fname)
        T = get_num_frames(path)
        frame_counts[T] += 1

    print("\n===== СТАТИСТИКА ПО КОЛИЧЕСТВУ КАДРОВ =====")
    for frames, count in sorted(frame_counts.items()):
        print(f"{frames:4d} кадров → {count} семплов")

    print("\nВсего семплов:", sum(frame_counts.values()))



In [ ]:
#process_folder_num_frames("../../data/dataset/keypoints")

Reading PKLs: 100%|██████████| 48/48 [00:00<00:00, 26875.80it/s]


===== СТАТИСТИКА ПО КОЛИЧЕСТВУ КАДРОВ =====
  43 кадров → 1 семплов
  44 кадров → 4 семплов
  45 кадров → 2 семплов
  46 кадров → 3 семплов
  47 кадров → 11 семплов
  48 кадров → 9 семплов
  49 кадров → 5 семплов
  50 кадров → 1 семплов
  51 кадров → 1 семплов
  52 кадров → 2 семплов
  53 кадров → 2 семплов
  54 кадров → 2 семплов
  55 кадров → 1 семплов
  56 кадров → 1 семплов
  57 кадров → 1 семплов
  59 кадров → 1 семплов
  62 кадров → 1 семплов

Всего семплов: 48


## Augmentations over folder

In [ ]:
def random_rotation(kps, max_deg=15):
    deg = np.random.uniform(-max_deg, max_deg)
    rad = np.deg2rad(deg)
    rot = np.array([[np.cos(rad), -np.sin(rad)],
                    [np.sin(rad),  np.cos(rad)]])
    return kps @ rot


def random_scale(kps, scale_range=(0.9, 1.1)):
    s = np.random.uniform(scale_range[0], scale_range[1])
    return kps * s


def random_shift(kps, shift_range=0.1):
    shift = np.random.uniform(-shift_range, shift_range, size=(1, 1, 2))
    return kps + shift


def add_noise(kps, sigma=0.01):
    return kps + np.random.normal(0, sigma, kps.shape)


def augment_keypoints(keypoint):
    """keypoint: (P, T, V, 2)"""
    k = keypoint.copy()

    # применяем аугментации
    k = random_rotation(k)
    k = random_scale(k)
    k = random_shift(k)
    k = add_noise(k)

    return k


def augment_folder(folder, num_aug=3):
    files = [f for f in os.listdir(folder) if f.endswith('.pkl')]

    for fname in tqdm(files, desc="Augmenting"):
        path = os.path.join(folder, fname)

        with open(path, 'rb') as f:
            data = pickle.load(f)

        for i in range(num_aug):
            new_data = data.copy()
            new_kp = augment_keypoints(data['keypoint'])

            new_data['keypoint'] = new_kp

            # генерируем имя: 001A003.pkl → 001A003_aug1.pkl
            base, ext = os.path.splitext(fname)
            new_name = f"{base}_aug{i}{ext}"
            new_path = os.path.join(folder, new_name)

            with open(new_path, 'wb') as f:
                pickle.dump(new_data, f, protocol=pickle.HIGHEST_PROTOCOL)


In [ ]:
#augment_folder("../../data/dataset/keypoints", num_aug=10)

Augmenting: 100%|██████████| 48/48 [00:00<00:00, 312.21it/s]


## Build dataset from folder

In [ ]:
def load_annotation(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def parse_label(filename):
    """
    Извлекает label по шаблону Axxx
    1A001.pkl → label = 0
    """
    base = os.path.splitext(filename)[0]
    import re
    m = re.search(r"A(\d{3})", base)
    if m is None:
        raise ValueError(f"Cannot extract class from filename: {filename}")
    cls = int(m.group(1)) - 1
    return cls


def parse_frame_dir(filename):
    """ Имя файла без расширения """
    return os.path.splitext(filename)[0]


def build_dataset(folder, out_path, val_ratio=0.2, seed=42):
    random.seed(seed)

    files = [f for f in os.listdir(folder) if f.endswith(".pkl")]

    annotations = []
    per_class = defaultdict(list)

    print("Loading .pkl annotations...")
    for fname in tqdm(files):
        path = os.path.join(folder, fname)

        ann = load_annotation(path)
        label = parse_label(fname)
        frame_dir = parse_frame_dir(fname)

        ann["label"] = label
        ann["frame_dir"] = frame_dir

        annotations.append(ann)
        per_class[label].append(frame_dir)

    # ---- Формируем валидацию: минимум 1 семпл на класс ----
    val_ids = []
    train_ids = []

    for cls, vids in per_class.items():
        random.shuffle(vids)

        # минимум один валидационный
        val_ids.append(vids[0])

        # остальные — в train
        train_ids.extend(vids[1:])

    # ---- Дополняем валидацию до заданного размера (val_ratio) ----
    total = len(train_ids) + len(val_ids)
    target_val_count = int(total * val_ratio)

    remaining_train = train_ids.copy()
    random.shuffle(remaining_train)

    while len(val_ids) < target_val_count and remaining_train:
        val_ids.append(remaining_train.pop())

    # оставшиеся — final train
    train_ids = [vid for vid in train_ids if vid not in val_ids]

    random.shuffle(train_ids)
    random.shuffle(val_ids)

    dataset = {
        "split": {
            "train": train_ids,
            "val": val_ids
        },
        "annotations": annotations
    }

    # ---- Save ----
    if out_path.endswith(".pkl"):
        with open(out_path, "wb") as f:
            pickle.dump(dataset, f, protocol=pickle.HIGHEST_PROTOCOL)

    elif out_path.endswith(".json"):
        def np_convert(o):
            import numpy as np
            if isinstance(o, np.ndarray):
                return o.tolist()
            raise TypeError

        with open(out_path, "w") as f:
            json.dump(dataset, f, default=np_convert)

    print(f"\nDataset saved to {out_path}")
    print(f"Train: {len(train_ids)} samples")
    print(f"Val:   {len(val_ids)} samples")
    print(f"Num classes: {len(per_class)}")


In [ ]:
#build_dataset("../../data/dataset/keypoints", "../../data/dataset/dataset.pkl")

Loading .pkl annotations...


100%|██████████| 528/528 [00:00<00:00, 20467.77it/s]


Dataset saved to ../../data/dataset/dataset.pkl
Train: 422 samples
Val:   106 samples
